# E-Commerce Descriptive Statistics

**Dataset:** Online Retail Transactions (UCI)  
**Purpose:** Deep statistical analysis of the same dataset used in `EDA.ipynb`.  
This notebook assumes the data is already understood — no cleaning steps are repeated.  
The focus is on quantifying distributional properties to guide feature engineering and modeling.

---

## 1. Setup & Imports

Load all libraries, configure plot aesthetics, and read the same CSV used in the EDA.  
A shape check and `.head()` confirm the data loaded correctly.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (11, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 40)

print('Libraries loaded.')

In [ ]:
# Locate the CSV
_candidates = [
    os.path.join('..', 'data', 'raw', 'data.csv'),
    os.path.join('ecommerce_analysis', 'data', 'raw', 'data.csv'),
    os.path.join('..', '..', 'data', 'raw', 'data.csv'),
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError('data.csv not found — adjust _candidates above.')

df_raw = pd.read_csv(DATA_PATH, encoding='ISO-8859-1', low_memory=False)
print(f'Loaded  {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Path    : {os.path.abspath(DATA_PATH)}')

In [ ]:
df = df_raw.copy()

DATE_COL = None
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col], infer_datetime_format=True)
            DATE_COL = col
            break
        except Exception:
            pass

num_cols_raw = df.select_dtypes(include='number').columns.tolist()
QTY_COL   = next((c for c in num_cols_raw if 'qty' in c.lower() or 'quant' in c.lower()), None)
PRICE_COL = next((c for c in num_cols_raw if 'price' in c.lower() or 'unit' in c.lower()), None)

if QTY_COL and PRICE_COL:
    df['Revenue'] = df[QTY_COL] * df[PRICE_COL]
    print(f'Revenue = {QTY_COL} x {PRICE_COL}')

INVOICE_COL  = next((c for c in df.columns if 'invoice' in c.lower()), None)
CUSTOMER_COL = next((c for c in df.columns if 'customer' in c.lower() or 'cust' in c.lower()), None)
COUNTRY_COL  = next((c for c in df.columns if 'country' in c.lower()), None)
DESC_COL     = next((c for c in df.columns if 'desc' in c.lower()), None)

NUMERIC_COLS = df.select_dtypes(include='number').columns.tolist()
CAT_COLS     = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric : {NUMERIC_COLS}')
print(f'Categorical: {CAT_COLS}')
print(f'Date col: {DATE_COL}')

In [ ]:
display(df.head(8))


---
## 2. Central Tendency Measures

Central tendency describes where the center of a distribution falls.

| Measure | Best used when |
|---------|----------------|
| **Mean** | Distribution is roughly symmetric with no extreme outliers |
| **Median** | Distribution is skewed or contains outliers (more robust) |
| **Mode** | Discrete or categorical data, or to find the most common value |

For e-commerce data, the **median** is usually more informative because a handful of
large wholesale orders can dramatically inflate the mean.

In [ ]:
central = pd.DataFrame(index=NUMERIC_COLS)
central['mean']   = df[NUMERIC_COLS].mean()
central['median'] = df[NUMERIC_COLS].median()
central['mode']   = df[NUMERIC_COLS].apply(
    lambda s: s.mode().iloc[0] if not s.mode().empty else float('nan')
)
central['mean_vs_median_gap'] = (central['mean'] - central['median']).abs()
central['skew_indicator'] = central.apply(
    lambda r: 'right-skewed (mean > median)' if r['mean'] > r['median']
    else ('left-skewed (mean < median)' if r['mean'] < r['median'] else 'symmetric'),
    axis=1
)

display(central)

**Reading the table:** A large `mean_vs_median_gap` signals a skewed distribution.
Prefer the median for such columns when reporting a typical value.
Columns where the mean far exceeds the median are prime candidates for log-transformation.

---
## 3. Dispersion Measures

Dispersion quantifies how spread out the data is around its center.

- **Range**: Max minus min — sensitive to extreme values.
- **Variance / Std Dev**: Average squared deviation from the mean (Std Dev is in original units).
- **IQR**: The middle 50% spread — robust to outliers.
- **CV (Coefficient of Variation)**: Std Dev / |Mean| — compares variability across different scales.

High CV in revenue or price suggests a wide product price range or heterogeneous customer segments.

In [ ]:
disp = pd.DataFrame(index=NUMERIC_COLS)
disp['min']      = df[NUMERIC_COLS].min()
disp['max']      = df[NUMERIC_COLS].max()
disp['range']    = disp['max'] - disp['min']
disp['variance'] = df[NUMERIC_COLS].var()
disp['std_dev']  = df[NUMERIC_COLS].std()
disp['IQR']      = df[NUMERIC_COLS].apply(lambda s: s.quantile(0.75) - s.quantile(0.25))
disp['CV (%)']   = (disp['std_dev'] / df[NUMERIC_COLS].mean().abs() * 100).round(2)

display(disp)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
cv_sorted = disp['CV (%)'].sort_values(ascending=False)
cv_sorted.plot(kind='bar', ax=ax,
               color=sns.color_palette('coolwarm', len(cv_sorted)),
               edgecolor='white')
ax.set_title('Coefficient of Variation (CV) by Column')
ax.set_xlabel('Column')
ax.set_ylabel('CV (%)')
ax.tick_params(axis='x', rotation=30)
for p in ax.patches:
    ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5,
            f'{p.get_height():.0f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Shape of Distributions

- **Skewness**: Measures asymmetry. Positive = long right tail; negative = long left tail.  
  |skewness| > 1 is highly skewed; |skewness| < 0.5 is approximately symmetric.
- **Kurtosis** (excess): Measures tail heaviness relative to a normal distribution.  
  Positive (leptokurtic) = heavier tails; negative (platykurtic) = lighter tails.

For e-commerce, revenue and quantity are almost always positively skewed due to the mix
of everyday small purchases and occasional large bulk orders.

In [ ]:
shape = pd.DataFrame(index=NUMERIC_COLS)
shape['skewness'] = df[NUMERIC_COLS].skew()
shape['kurtosis'] = df[NUMERIC_COLS].kurt()  # excess kurtosis (normal = 0)
shape['skew_label'] = shape['skewness'].apply(
    lambda s: 'highly right-skewed' if s > 1
    else ('moderately right-skewed' if s > 0.5
    else ('approximately symmetric' if abs(s) <= 0.5
    else ('moderately left-skewed' if s > -1
    else 'highly left-skewed')))
)
shape['kurt_label'] = shape['kurtosis'].apply(
    lambda k: 'heavy-tailed (leptokurtic)' if k > 1
    else ('light-tailed (platykurtic)' if k < -1
    else 'near-normal tails')
)
display(shape)

In [ ]:
plot_num = [c for c in NUMERIC_COLS if df[c].nunique() > 5]
n_cols_h = 2
n_rows_h = (len(plot_num) + 1) // n_cols_h
fig, axes = plt.subplots(n_rows_h, n_cols_h, figsize=(14, 4 * n_rows_h))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, col in enumerate(plot_num):
    data = df[col].dropna()
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    clipped = data.clip(p1, p99)
    sk, ku = data.skew(), data.kurt()
    axes[i].hist(clipped, bins=60, color='steelblue',
                 edgecolor='white', alpha=0.65, density=True)
    clipped.plot.kde(ax=axes[i], color='tomato', linewidth=2, label='KDE')
    axes[i].set_title(f'{col}\nskew={sk:.2f}  kurt={ku:.2f}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions with KDE — clipped to 1st-99th percentile', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Percentiles & Quantiles

Percentiles divide sorted data into 100 equal parts. Key percentiles:

- **P5 / P95**: Capture typical transactions while trimming extremes.
- **P25 / P75** (Q1 / Q3): Define the IQR — the middle 50%.
- **P50** (Median): The robust central value.

The **P95/P50 ratio** reveals how extreme the top transactions are relative to a typical one.
A ratio of 10x means the 95th-percentile order is 10x larger than the median.

In [ ]:
pcts = [0.05, 0.25, 0.50, 0.75, 0.95]
pct_labels = ['P5', 'P25 (Q1)', 'P50 (Median)', 'P75 (Q3)', 'P95']

pct_df = df[NUMERIC_COLS].quantile(pcts).T
pct_df.columns = pct_labels
pct_df['P95/P50 ratio'] = (
    pct_df['P95'] / pct_df['P50 (Median)'].replace(0, float('nan'))
).round(2)

display(pct_df)

In [ ]:
n_cols_b = 2
n_rows_b = (len(plot_num) + 1) // n_cols_b
fig, axes = plt.subplots(n_rows_b, n_cols_b, figsize=(14, 4 * n_rows_b))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, col in enumerate(plot_num):
    data = df[col].dropna()
    clipped = data.clip(data.quantile(0.001), data.quantile(0.999))
    axes[i].boxplot(
        clipped, vert=True, patch_artist=True,
        boxprops=dict(facecolor='lightsteelblue', color='navy'),
        medianprops=dict(color='tomato', linewidth=2),
        whiskerprops=dict(color='navy'),
        capprops=dict(color='navy'),
        flierprops=dict(marker='o', color='grey', alpha=0.3, markersize=3)
    )
    q1, med, q3 = data.quantile(0.25), data.median(), data.quantile(0.75)
    axes[i].set_title(f'{col}\nQ1={q1:.2f}  Median={med:.2f}  Q3={q3:.2f}')
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots — Spread and Outliers (0.1%-99.9% clipped for display)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Categorical Columns Summary

For categorical variables we measure:

- **Frequency / Relative Frequency**: How often each category appears.
- **Mode**: The most frequent value — useful as a baseline imputation strategy.
- **Cardinality**: Number of unique values — high-cardinality columns need
  embedding or hashing before modeling.

In e-commerce, `Country` concentration and `Description` breadth are key signals
for market segmentation and inventory planning.

In [ ]:
cat_summary = pd.DataFrame({
    'unique_values':  df[CAT_COLS].nunique(),
    'pct_unique (%)': (df[CAT_COLS].nunique() / len(df) * 100).round(2),
    'null_count':     df[CAT_COLS].isna().sum(),
    'null_pct (%)':   (df[CAT_COLS].isna().mean() * 100).round(2),
    'mode':           df[CAT_COLS].apply(
        lambda s: s.mode().iloc[0] if not s.mode().empty else 'N/A'
    ),
    'mode_freq': df[CAT_COLS].apply(
        lambda s: s.value_counts().iloc[0] if len(s.value_counts()) > 0 else 0
    ),
    'mode_pct (%)': df[CAT_COLS].apply(
        lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 2)
        if len(s.value_counts()) > 0 else 0
    ),
}).sort_values('unique_values', ascending=False)

display(cat_summary)

In [ ]:
TOP_N = 15
plot_cat = [c for c in CAT_COLS if df[c].nunique() <= 200]

for col in plot_cat:
    freq = df[col].value_counts().head(TOP_N).reset_index()
    freq.columns = [col, 'count']
    freq['relative_freq (%)'] = (freq['count'] / len(df) * 100).round(2)
    freq['cumulative (%)']    = freq['relative_freq (%)'].cumsum().round(2)
    print(f'\n--- {col} | top {TOP_N} of {df[col].nunique()} unique values ---')
    display(freq)

In [ ]:
for col in plot_cat:
    counts = df[col].value_counts().head(TOP_N)
    fig, ax = plt.subplots(figsize=(10, 4))
    counts.plot(kind='bar', ax=ax,
                color=sns.color_palette('muted', len(counts)),
                edgecolor='white')
    ax.set_title(f'Top {TOP_N} Categories — {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=40)
    for p in ax.patches:
        ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 20,
                f'{int(p.get_height()):,}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

---
## 7. Grouped Descriptive Statistics

Grouping by a categorical variable reveals how numerical distributions differ across segments.
In e-commerce this answers questions like:
- *Do customers from different countries spend more per transaction?*
- *Which segment has the highest revenue variability?*

The mean/median gap within each group exposes within-segment skew, and std reveals
which segments are most heterogeneous. We focus on the top 10 groups by row volume.

In [ ]:
group_col = COUNTRY_COL or next((c for c in CAT_COLS if 2 < df[c].nunique() <= 50), None)
target_num = [c for c in NUMERIC_COLS if c != CUSTOMER_COL]

if group_col and target_num:
    top_groups = df[group_col].value_counts().head(10).index
    sub = df[df[group_col].isin(top_groups)]

    grouped = sub.groupby(group_col)[target_num].agg(['mean', 'median', 'std']).round(2)
    grouped.columns = ['_'.join(c) for c in grouped.columns]
    grouped = grouped.sort_values(f'{target_num[0]}_mean', ascending=False)

    print(f'Grouped by: {group_col}  |  Top 10 groups by row count')
    display(grouped)
else:
    print('No suitable grouping column found.')

In [ ]:
if group_col and 'Revenue' in df.columns:
    top_groups = df[group_col].value_counts().head(10).index
    sub = df[df[group_col].isin(top_groups)]
    pivot = sub.groupby(group_col)['Revenue'].agg(['mean', 'median']).sort_values('mean', ascending=False)

    fig, ax = plt.subplots(figsize=(11, 5))
    x = list(range(len(pivot)))
    w = 0.35
    ax.bar([i - w/2 for i in x], pivot['mean'],   w,
           label='Mean Revenue',   color='steelblue', edgecolor='white')
    ax.bar([i + w/2 for i in x], pivot['median'], w,
           label='Median Revenue', color='tomato',    edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=35, ha='right')
    ax.set_title(f'Mean vs Median Revenue per Transaction by {group_col}')
    ax.set_ylabel('Revenue (unit currency)')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 8. Outlier Detection Summary

We use the **IQR fence method** to flag outliers:

- **Lower fence** = Q1 - 1.5 x IQR  
- **Upper fence** = Q3 + 1.5 x IQR  
- Values outside these bounds are flagged as outliers.

**Impact on statistics:**  
Outliers inflate the mean, variance, and skewness. Handling options:
- **Winsorization**: Cap values at the fence thresholds.
- **Log-transformation**: Compress the range of extreme values.
- **Separate modeling**: Treat extreme segments (e.g., wholesale buyers) independently.

In [ ]:
outlier_rows = []
for col in NUMERIC_COLS:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_low  = int((data < lower).sum())
    n_high = int((data > upper).sum())
    n_tot  = n_low + n_high
    outlier_rows.append({
        'column':          col,
        'Q1':              round(float(Q1), 4),
        'Q3':              round(float(Q3), 4),
        'IQR':             round(float(IQR), 4),
        'lower_fence':     round(float(lower), 4),
        'upper_fence':     round(float(upper), 4),
        'outliers_low':    n_low,
        'outliers_high':   n_high,
        'total_outliers':  n_tot,
        'outlier_pct (%)': round(n_tot / len(data) * 100, 2)
    })

outlier_df = (
    pd.DataFrame(outlier_rows)
    .set_index('column')
    .sort_values('outlier_pct (%)', ascending=False)
)
display(outlier_df)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
outlier_df['outlier_pct (%)'].sort_values(ascending=True).plot(
    kind='barh', ax=ax, color='salmon', edgecolor='white'
)
ax.set_title('Outlier Rate by Column (IQR Fence Method)')
ax.set_xlabel('Outlier (%)')
ax.set_ylabel('Column')
for p in ax.patches:
    ax.text(p.get_width() + 0.2, p.get_y() + p.get_height() / 2,
            f'{p.get_width():.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
for col in plot_num:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    is_out = (data < lower) | (data > upper)

    sample = data.sample(min(5000, len(data)), random_state=42)
    flags  = is_out.loc[sample.index]

    fig, ax = plt.subplots(figsize=(11, 2.5))
    ax.scatter(range(len(sample)), sample.values,
               c=flags.map({True: 'tomato', False: 'steelblue'}),
               s=8, alpha=0.5)
    ax.axhline(lower, color='red',    linestyle='--', lw=1,
               label=f'Lower fence = {lower:.2f}')
    ax.axhline(upper, color='orange', linestyle='--', lw=1,
               label=f'Upper fence = {upper:.2f}')
    ax.set_title(
        f'{col} - outliers in red '
        f'({is_out.sum():,} = {is_out.mean()*100:.1f}%)'
    )
    ax.set_xlabel('Sample index')
    ax.set_ylabel(col)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 9. Key Takeaways

The cell below auto-generates a concise statistical summary grounded in the analysis above.
These findings directly inform feature engineering and modeling decisions.

In [ ]:
print('=' * 65)
print('  DESCRIPTIVE STATISTICS - KEY FINDINGS')
print('=' * 65)

most_skewed = shape['skewness'].abs().idxmax()
print(f'\n1. Highest skewness   : {most_skewed} '
      f'(skew = {shape.loc[most_skewed, "skewness"]:.2f}) '
      f'-> {shape.loc[most_skewed, "skew_label"]}')

high_cv = disp['CV (%)'].abs().idxmax()
print(f'2. Highest variability: {high_cv} '
      f'(CV = {disp.loc[high_cv, "CV (%)"]:.0f}%) '
      f'- wide spread, consider log-transform or winsorization')

if 'Revenue' in central.index:
    rm   = central.loc['Revenue', 'mean']
    rmed = central.loc['Revenue', 'median']
    ratio = rm / rmed if rmed != 0 else float('nan')
    print(f'3. Revenue mean vs median: {rm:.2f} vs {rmed:.2f} '
          f'(ratio {ratio:.1f}x) - right-skewed by large wholesale orders')

worst_out = outlier_df['outlier_pct (%)'].idxmax()
print(f'4. Most outliers      : {worst_out} '
      f'({outlier_df.loc[worst_out, "outlier_pct (%)"]:.1f}% flagged) '
      f'- includes returns and bulk orders')

if COUNTRY_COL:
    top_pct = df[COUNTRY_COL].value_counts(normalize=True).iloc[0] * 100
    top_val = df[COUNTRY_COL].value_counts().index[0]
    print(f'5. {COUNTRY_COL} concentration: "{top_val}" = {top_pct:.1f}% of rows - heavily imbalanced')

high_kurt = shape['kurtosis'].abs().idxmax()
print(f'6. Heaviest tails     : {high_kurt} '
      f'(excess kurtosis = {shape.loc[high_kurt, "kurtosis"]:.1f}) '
      f'- extreme values more common than in a normal distribution')

print('\n7. Modeling recommendations:')
print('   - Log-transform or winsorize highly skewed / high-CV columns before regression')
print('   - Use median (not mean) when reporting typical transaction values')
print('   - Exclude or model separately: negative-quantity rows (returns), zero-price rows')
print('   - High-cardinality categoricals need embedding or hashing encoding')
print('   - Confirm normality assumptions before applying parametric tests')

print('\n' + '=' * 65)